# 05 — ETL v5: queries de voz-de-víctima (DIAGNÓSTICA)

Rediseño de las queries para resolver el sesgo crypto/ponzi detectado en la v4:
- v4: 92% de tweets eran crypto-discusión (Ponzi como insulto político/retórico).
- v5: queries exigen patrones de **voz de víctima** o **alerta comunitaria** en origen.

## Cambios clave

1. **Quitada `ponzi` suelta** — era palabra-imán para insultos políticos.
2. **Cada query exige co-ocurrencia con patrón de víctima/testigo:**
   - "I lost...", "they took...", "sent money...", "scammed me..."
   - "got a text/email/call", "received...", "anyone else...", "watch out for..."
3. **Sin `crypto scam` suelta** — ahora va con verbo de víctima.

## ⚠️ ESTA ES UNA TIRADA DIAGNÓSTICA

`PAGES_PER_QUERY = 1` por defecto. 5 llamadas a la API.

El objetivo es ver qué calidad y volumen dan las nuevas queries antes de la tirada definitiva.


In [ ]:
# Imports y carga del token
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv

pd.set_option('display.max_colwidth', None)

load_dotenv()
TOKEN = os.getenv("X_BEARER_TOKEN", "")
if not TOKEN:
    raise ValueError("Falta X_BEARER_TOKEN en .env")

BASE_URL = "https://api.x.com/2/tweets/search/all"


In [ ]:
def search_all_posts_to_df(
    bearer_token, query, max_pages=1, max_results=500,
    sleep_seconds=1.1, start_time=None, end_time=None, label=""
):
    headers = {"Authorization": f"Bearer {bearer_token}"}
    params = {
        "query": query,
        "max_results": max_results,
        "tweet.fields": "id,text,created_at,author_id,lang,geo,public_metrics,entities",
        "expansions": "author_id,geo.place_id",
        "user.fields": "id,name,username,created_at,location,verified,public_metrics",
        "place.fields": "id,full_name,country,country_code,place_type,geo",
        "sort_order": "recency",
    }
    if start_time: params["start_time"] = start_time
    if end_time: params["end_time"] = end_time

    all_rows = []
    next_token = None

    for page in range(max_pages):
        if next_token:
            params["next_token"] = next_token
        else:
            params.pop("next_token", None)

        retries = 0
        while True:
            r = requests.get(BASE_URL, headers=headers, params=params, timeout=30)
            if r.status_code == 200:
                break
            elif r.status_code == 429:
                wait = min(60 * (2 ** retries), 900)
                print(f"  Rate limit. Esperando {wait}s...")
                time.sleep(wait)
                retries += 1
                if retries > 5: raise RuntimeError("Demasiados reintentos")
            elif r.status_code == 400:
                raise RuntimeError(f"400 Bad Request. Query len={len(query)}. {r.text}")
            elif r.status_code == 403:
                raise RuntimeError(f"403 Forbidden. {r.text}")
            else:
                raise RuntimeError(f"X API error {r.status_code}: {r.text}")

        payload = r.json()
        data = payload.get("data", []) or []
        includes = payload.get("includes", {}) or {}
        meta = payload.get("meta", {}) or {}
        users_by_id = {u["id"]: u for u in includes.get("users", []) or []}
        places_by_id = {p["id"]: p for p in includes.get("places", []) or []}

        for t in data:
            author = users_by_id.get(t.get("author_id"), {})
            place_id = (t.get("geo") or {}).get("place_id")
            place = places_by_id.get(place_id, {}) if place_id else {}
            entities = t.get("entities") or {}
            metrics = t.get("public_metrics") or {}
            am = author.get("public_metrics") or {}

            all_rows.append({
                "tweet_id": t.get("id"),
                "created_at": t.get("created_at"),
                "text": t.get("text"),
                "lang": t.get("lang"),
                "author_id": t.get("author_id"),
                "username": author.get("username"),
                "name": author.get("name"),
                "user_location": author.get("location"),
                "user_verified": author.get("verified"),
                "user_followers": am.get("followers_count"),
                "user_tweet_count": am.get("tweet_count"),
                "place_id": place_id,
                "place_full_name": place.get("full_name"),
                "place_country": place.get("country"),
                "place_country_code": place.get("country_code"),
                "place_type": place.get("place_type"),
                "retweet_count": metrics.get("retweet_count"),
                "reply_count": metrics.get("reply_count"),
                "like_count": metrics.get("like_count"),
                "quote_count": metrics.get("quote_count"),
                "n_hashtags": len(entities.get("hashtags", []) or []),
                "n_mentions": len(entities.get("mentions", []) or []),
                "n_urls": len(entities.get("urls", []) or []),
                "query_label": label,
            })

        print(f"  [{label} | p{page+1}/{max_pages}] {len(all_rows)} acumulados")
        next_token = meta.get("next_token")
        if not next_token:
            print(f"  [{label}] sin más páginas")
            break
        time.sleep(sleep_seconds)

    return pd.DataFrame(all_rows)


## Queries v5: voz-de-víctima

Todas exigen un patrón de fraude + un patrón de víctima/testigo.


In [ ]:
EX_MIN = "-trump -biden -election -voter -ballot -nba -nfl -kpop"
GEO_OPS = "lang:en place_country:US -is:retweet -is:reply -is:nullcast"

def build(positive):
    q = f"({positive}) {EX_MIN} {GEO_OPS}"
    if len(q) > 512:
        raise ValueError(f"Query excede 512 chars: {len(q)}")
    return q

QUERIES = {
    "phishing": build(
        '"got a text" OR "got an email" OR "got a call" OR "received a text" '
        'OR "received an email" OR "fake text from" OR "fake email from" '
        'OR "phishing text" OR "phishing email" OR "scam text" OR "scam email"'
    ),
    "payment_apps": build(
        '("Zelle scam" OR "Venmo scam" OR "Cash App scam" OR "PayPal scam") '
        '("lost" OR "stolen" OR "I sent" OR "they took" OR "scammed me" '
        'OR "won\'t refund" OR "frozen account")'
    ),
    "crypto": build(
        '("rug pull" OR "pig butchering" OR "crypto scam" OR "bitcoin scam") '
        '("lost" OR "scammed me" OR "I sent" OR "stole my" OR "drained my" '
        'OR "fake exchange" OR "fake wallet")'
    ),
    "romance": build(
        '("romance scam" OR "dating scam" OR "pig butchering") '
        '("sent money" OR "sent him" OR "sent her" OR "wired" OR "lost" '
        'OR "scammed me" OR "asked for money")'
    ),
    "impersonation": build(
        '("IRS scam" OR "USPS scam" OR "FedEx scam" OR "toll scam" '
        'OR "fake delivery" OR "tech support scam") '
        '("got a" OR "received" OR "called me" OR "texted me" '
        'OR "anyone else" OR "watch out" OR "warning")'
    ),
}

print("=== QUERIES v5 ===\n")
for label, q in QUERIES.items():
    print(f"{label:<15} {len(q):>4} chars")
    print(f"  {q}\n")


## Configuración de la tirada (DIAGNÓSTICA por defecto)

In [ ]:
# ⚠️ TIRADA DIAGNÓSTICA
PAGES_PER_QUERY = 1     # 1 = diagnóstica (~2.500 max) | 8 = definitiva (~20.000)

START = "2025-01-01T00:00:00Z"
END   = "2025-12-31T23:59:59Z"

est_calls = len(QUERIES) * PAGES_PER_QUERY
est_max_tweets = est_calls * 500
print(f"Llamadas estimadas:  {est_calls}")
print(f"Tweets máx:          {est_max_tweets}")
print(f"Ventana:             {START} → {END}")
print()
input("Pulsa ENTER para ejecutar, o Ctrl+C para abortar... ")


## Ejecución de las 5 queries

In [ ]:
dfs = []
for label, query in QUERIES.items():
    print(f"\n>>> {label}")
    df_part = search_all_posts_to_df(
        bearer_token=TOKEN, query=query,
        max_pages=PAGES_PER_QUERY, max_results=500,
        sleep_seconds=1.1, start_time=START, end_time=END,
        label=label,
    )
    print(f">>> {label}: {len(df_part)}")
    dfs.append(df_part)

df_raw = pd.concat(dfs, ignore_index=True)
print(f"\n=== TOTAL BRUTO: {len(df_raw)} ===")

# Distribución por categoría — info clave para comparar con v4
print("\nPor categoría:")
print(df_raw["query_label"].value_counts())


## Deduplicación

In [ ]:
df_raw["query_labels"] = df_raw.groupby("tweet_id")["query_label"].transform(
    lambda s: ",".join(sorted(set(s)))
)
df_dedup = df_raw.drop_duplicates(subset="tweet_id", keep="first").reset_index(drop=True)
df_dedup = df_dedup.drop(columns=["query_label"])
print(f"Únicos tras deduplicar: {len(df_dedup)}")
print(f"Multi-categoría: {(df_dedup['query_labels'].str.contains(',')).sum()}")


## Filtros pandas (los buenos del v4: recovery scammers, blog promos, metáfora institucional, etc.)

In [ ]:
import re

STRONG_FRAUD_TERMS = re.compile(
    r"\b(scam|scammed|scammer|scammers|fraud|fraudster|defrauded|"
    r"phishing|smishing|vishing|ponzi|rug\s*pull|pig\s*butchering|"
    r"identity\s*theft|wire\s*fraud|impersonat|spoof|"
    r"fake\s*(?:text|email|delivery|invoice|website|caller))\b",
    re.IGNORECASE,
)

MONEY_TERMS = re.compile(
    r"(\b(money|cash|dollar|dollars|usd|paid|payment|sent|wired|wire|"
    r"transfer|refund|bank|account|credit\s*card|debit|wallet|invoice|"
    r"charged|stolen|lost|victim|defrauded|deposit|withdraw|"
    r"check|venmo|zelle|paypal|crypto|bitcoin|coinbase|gift\s*card)\b|"
    r"\$\s*\d+|\d+\s*(k|dollars|usd))",
    re.IGNORECASE,
)

RECOVERY_SCAM_PATTERNS = re.compile(
    r"\b(DM\s+(now|me|us|asap)|we\s+are\s+tracing|get\s+your\s+money\s+back|"
    r"recover(y|\s+expert|\s+specialist)|funds\s+recovery|crypto\s+recovery|"
    r"contact\s+us\s+(?:now|asap|today)|reach\s+out\s+to\s+(?:us|me)|"
    r"100%\s+(?:guaranteed|recovery|success))\b",
    re.IGNORECASE,
)

BRAND_LIST_PATTERN = re.compile(
    r"(nft\s+scam|bitcoin\s+scam|forex|cfd|binary\s+option|"
    r"wallet\s+drained|bored\s*apes?|ethereum\s+scam)",
    re.IGNORECASE,
)

def is_recovery_scammer(text):
    if not isinstance(text, str): return False
    if RECOVERY_SCAM_PATTERNS.search(text): return True
    has_contact = bool(re.search(r"\b(DM|dm)\b|contact|reach\s+out", text, re.IGNORECASE))
    return has_contact and len(BRAND_LIST_PATTERN.findall(text)) >= 2

PROMO_BLOG_PATTERN = re.compile(
    r"\b(our\s+(?:recent\s+)?(?:blog|article|post)|breaks?\s+down|"
    r"read\s+(?:more|the\s+full)|link\s+in\s+bio|"
    r"check\s+(?:out|it)\s+(?:our|this)|"
    r"new\s+(?:blog|article|episode|video))\b",
    re.IGNORECASE,
)
NEWS_HEADLINE_PATTERN = re.compile(
    r"^\W*(leaders?|chief|ceo|president|founder|judge|jury|court|"
    r"police|sentence[d]?|charged|arrested|indicted|"
    r"\d+\s+(?:years?|months?)\s+in\s+prison)\b",
    re.IGNORECASE,
)
def looks_like_news_or_promo(text, n_urls=0):
    if not isinstance(text, str): return False
    if PROMO_BLOG_PATTERN.search(text): return True
    if n_urls > 0 and NEWS_HEADLINE_PATTERN.search(text): return True
    return False

INSTITUTIONAL_TARGETS = [
    "social security", "stock market", "housing market", "real estate market",
    r"\bai\b", "artificial intelligence", "crypto industry",
    "the tax system", "tax system", "taxes", "capitalism", "the government",
    "modern slavery", "slave engine", "marriage", "religion", "the church",
    "college", "the system", "the economy", "the federal reserve",
    "fiat currency", "social welfare", "welfare system",
    "healthcare system", "education system", "school system",
    "pension", "401k", "wall street",
]
ACCUSATION_WORDS = ["ponzi", "scheme", "scam", "fraud"]
WINDOW = r"(?:\W+\w+){0,6}\W+"
INSTITUTIONAL_METAPHOR_RE = re.compile(
    "|".join(
        f"({inst}{WINDOW}(?:{'|'.join(ACCUSATION_WORDS)}))"
        f"|((?:{'|'.join(ACCUSATION_WORDS)}){WINDOW}{inst})"
        for inst in INSTITUTIONAL_TARGETS
    ),
    re.IGNORECASE,
)
def is_institutional_metaphor(text):
    if not isinstance(text, str): return False
    return bool(INSTITUTIONAL_METAPHOR_RE.search(text))

METAPHOR_PATTERNS = re.compile(
    r"\b(life is a scam|love is a scam|system is a scam|"
    r"capitalism is a scam|college is a scam|dating is a scam|"
    r"my life is a scam|everything is a scam)\b",
    re.IGNORECASE,
)
SOFT_POLITICS_PATTERNS = re.compile(
    r"\b(harris|desantis|newsom|kamala|obama|congress|senate|senator|"
    r"republican|democrat|gop|maga|lawmaker|politician|"
    r"voter fraud|election fraud|stolen election|rigged election|"
    r"deep state)\b",
    re.IGNORECASE,
)
BRAND_USERNAMES = {
    "lifelock", "aura", "norton", "nortononline", "mcafee",
    "cnn", "foxnews", "msnbc", "nytimes", "wsj", "reuters",
    "ap", "bbcworld", "abc", "abcnews", "nbcnews", "cbsnews",
}
EMOTIONAL_ONLY = re.compile(
    r"\b(catfish|catfished|catfishing|emotional scam|"
    r"scammed my heart|broke my heart)\b",
    re.IGNORECASE,
)
URL_RE = re.compile(r"https?://\S+")
MENTION_RE = re.compile(r"@\w+")
def clean_for_length(text):
    if not isinstance(text, str): return ""
    t = URL_RE.sub("", text)
    t = MENTION_RE.sub("", t)
    return re.sub(r"\s+", " ", t).strip()

us_keywords = ["united states", "usa", "u.s.", " us ", " us,", "america",
    "new york", "california", "texas", "florida", "chicago",
    "los angeles", "boston", "seattle", "miami", "atlanta",
    "dallas", "houston", "philadelphia", "phoenix", "denver",
    "washington", "san francisco", "san diego"]
def looks_us(row):
    if row.get("place_country_code") == "US": return True
    loc = " " + str(row.get("user_location") or "").lower() + " "
    return any(k in loc for k in us_keywords)

# Aplicar
df = df_dedup.copy()
df["clean_text"]       = df["text"].apply(clean_for_length)
df["clean_len"]        = df["clean_text"].str.len()
df["n_words"]          = df["clean_text"].str.split().str.len().fillna(0).astype(int)
df["hashtag_ratio"]    = (df["n_hashtags"] / df["n_words"].replace(0, 1)).fillna(0)
df["mention_ratio"]    = (df["n_mentions"] / df["n_words"].replace(0, 1)).fillna(0)
df["is_metaphor"]      = df["text"].fillna("").apply(lambda t: bool(METAPHOR_PATTERNS.search(t)))
df["is_soft_politics"] = df["text"].fillna("").apply(lambda t: bool(SOFT_POLITICS_PATTERNS.search(t)))
df["is_emotional"]     = df["text"].fillna("").apply(lambda t: bool(EMOTIONAL_ONLY.search(t)))
df["is_brand_account"] = df["username"].fillna("").str.lower().isin(BRAND_USERNAMES)
df["likely_us"]        = df.apply(looks_us, axis=1)
df["has_strong_fraud"] = df["text"].fillna("").apply(lambda t: bool(STRONG_FRAUD_TERMS.search(t)))
df["has_money"]        = df["text"].fillna("").apply(lambda t: bool(MONEY_TERMS.search(t)))
df["is_recovery_scammer"] = df["text"].fillna("").apply(is_recovery_scammer)
df["is_news_or_promo"]    = df.apply(lambda r: looks_like_news_or_promo(r["text"], r.get("n_urls", 0)), axis=1)
df["is_institutional_metaphor"] = df["text"].fillna("").apply(is_institutional_metaphor)
df["semantically_relevant"] = df["has_strong_fraud"] | df["has_money"]

mask = (
    (df["clean_len"] >= 40) &
    (df["semantically_relevant"]) &
    (df["likely_us"]) &
    (df["hashtag_ratio"] < 0.3) &
    (df["mention_ratio"] < 0.4) &
    (~df["is_metaphor"]) &
    (~df["is_soft_politics"]) &
    (~df["is_emotional"]) &
    (~df["is_brand_account"]) &
    (~df["is_recovery_scammer"]) &
    (~df["is_news_or_promo"]) &
    (~df["is_institutional_metaphor"])
)
df_clean = df[mask].reset_index(drop=True)

print("=== RESUMEN v5 ===")
print(f"Brutos:                {len(df_raw)}")
print(f"Únicos:                {len(df_dedup)}")
print(f"Limpios finales:       {len(df_clean)}")
print(f"Retención sobre brutos: {len(df_clean)/max(len(df_raw),1)*100:.1f}%")
print()
print("Por categoría temática:")
for label in QUERIES.keys():
    n = df_clean["query_labels"].fillna("").str.contains(label).sum()
    print(f"  {label:<20} {n:>6}")


## Inspección visual

Muestra ALEATORIA y muestra POR CATEGORÍA — para detectar sesgos.


In [ ]:
print("=== MUESTRA ALEATORIA DE 20 ===\n")
for _, row in df_clean.sample(min(20, len(df_clean)), random_state=42).iterrows():
    print(f"[{row['query_labels']}] @{row['username']} — {row['user_location']}")
    print(f"  {row['text'][:280]}")
    print()


In [ ]:
# Muestra por categoría: 3 tweets de cada una para ver representatividad
print("=== 3 EJEMPLOS POR CATEGORÍA ===\n")
for label in QUERIES.keys():
    subset = df_clean[df_clean["query_labels"].fillna("").str.contains(label)]
    print(f"--- {label.upper()} ({len(subset)} tweets) ---")
    for _, row in subset.sample(min(3, len(subset)), random_state=42).iterrows():
        print(f"  @{row['username']}: {row['text'][:240]}")
    print()


## Guardado

In [ ]:
import os
os.makedirs("../data/raw", exist_ok=True)
df_dedup.to_csv("../data/raw/scam_us_raw_dedup_v5.csv", index=False, encoding="utf-8")
df_clean.to_csv("../data/raw/scam_us_clean_v5.csv", index=False, encoding="utf-8")
print(f"Guardados:")
print(f"  scam_us_raw_dedup_v5.csv  ({len(df_dedup)} filas)")
print(f"  scam_us_clean_v5.csv      ({len(df_clean)} filas)")
